In [ ]:
# q01
import numpy as np
import pandas as pd
def generate_constellations():
    constellations = {}

    # QPSK
    qpsk_coords = np.array([-0.5, 0.5])
    X_q, Y_q = np.meshgrid(qpsk_coords, qpsk_coords) # multiplies axes 
    constellations['QPSK'] = X_q.flatten() + 1j * Y_q.flatten()

    # 8-PSK
    """
    8 points, 360 degrees. 360/8 = 45. so each point is 45 degrees away. using dmin =1 and law of cosines (googled this), we solve for radius R = sqrt(1 / 2-sqrt(2)). to rotate, multiply by exp(jtheta)
    """
    R_8psk = np.sqrt(1 / (2 - np.sqrt(2)))
    angles = np.linspace(0, 2*np.pi, 8, endpoint=False) # endpoint = False bc repeats the first point at 2pi
    constellations['8-PSK'] = R_8psk * np.exp(1j * angles)

    # 16-QAM
    # a+jb where a and b are from the set 
    qam16_pts = [-1.5, -0.5, 0.5, 1.5]
    qam16_coords = np.array([complex(x, y) for x in qam16_pts for y in qam16_pts]) # nested list comprehension
    constellations['16-QAM'] = qam16_coords

    # 32-qam 6x6 grid excluding the 4 corners
    qam32_pts = [-2.5, -1.5, -0.5, 0.5, 1.5, 2.5]
    qam32_coords = np.array([complex(x, y) for x in qam32_pts for y in qam32_pts if not (abs(x) == 2.5 and abs(y) == 2.5)])
    constellations['32-QAM'] = qam32_coords
    print(constellations)
    # Eb = 1/k * Es = 1/log2(M) * Es 
    # Es = 1/M * sum(|s|^2) for all s in constellation

    results = []
    for name, pts in constellations.items():
        M = len(pts)
        Es = np.mean(np.abs(pts)**2) # Sum of squared distances / M
        Eb = Es / np.log2(M)         
        #eta = np.log2(M) / 2         
        
        results.append({
            'Constellation': name,
            'M': M,
            'E_s': round(Es, 3),
            'E_b/d_min^2': round(Eb, 3), # Since d_min = 1, E_b = E_b / d_min^2
            'N': round(np.log2(M) / 2, 3) # in phase dimension and quadrature dimension, N = log2(M)/2
        })
        
    return pd.DataFrame(results) # cool trick!


In [ ]:
df = generate_constellations()
df

In [ ]:
import matplotlib.pyplot as plt

A = 1
W = 1
T = 1 / (2*W)


f = np.linspace(-W*2, W*2, 1000)
rect = np.where(np.abs(f) <= W, 1.0, 0.0)

X_f = (A / (2 * W)) * (1 + np.cos(np.pi * f / W)) * rect  # from assignment

t= np.linspace(-T*2, T*2, 1000)
x_t = A * np.sinc(2* W * t) + (A/2) * np.sinc(2* W * (t+T)) + (A/2) * np.sinc(2* W * (t-T)) # calculated 

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(f, X_f, 'b', linewidth=2)
ax1.set_title('Spectrum $X(f)$')
ax1.set_xlabel('Frequency $f$')
ax1.set_ylabel('$X(f)$')
ax1.grid(True)
ax1.axhline(0, color='black', linewidth=0.5)
ax1.axvline(0, color='black', linewidth=0.5)


ax2.plot(t, x_t, 'r', linewidth=2)
ax2.set_title('Time Domain $x(t)$')
ax2.set_xlabel('Time $t$')
ax2.set_ylabel('$x(t)$')
ax2.grid(True)
ax2.axhline(0, color='black', linewidth=0.5)
ax2.axvline(0, color='black', linewidth=0.5)